In [2]:
from ngsolve import *
from netgen.csg import unit_cube
# 关键修改：使用 webgui 替代 netgen.gui
from ngsolve.webgui import Draw 

# 1. 物理参数
k = 10.0
source_pos = (0.5, 0.5, 0.5)

# 2. 生成网格
geo = unit_cube
mesh = Mesh(geo.GenerateMesh(maxh=0.1))

# 3. 定义复数 HCurl 空间 (电场 E 属于 H(curl))
fes = HCurl(mesh, order=2, complex=True)
E, v = fes.TnT()

# 4. 定义弱形式
a = BilinearForm(fes)
# 体积分项: (curl E * curl v - k^2 * E * v)
a += (curl(E) * curl(v) - k**2 * E * v) * dx
# 一阶吸收边界条件 (ABC): Silver-Müller
# 这里的 1j 是虚数单位 i
a += - 1j * k * E.Trace() * v.Trace() * ds

# 5. 定义源项 (在中心放置一个 X 方向的电流源)
f = LinearForm(fes)
peak = 100
source_func = peak * exp(-50 * ((x-0.5)**2 + (y-0.5)**2 + (z-0.5)**2))
f += CoefficientFunction((source_func, 0, 0)) * v * dx

# 6. 求解
gfu = GridFunction(fes)
with TaskManager():
    a.Assemble()
    f.Assemble()
    # 使用稀疏矩阵直接求解器
    inv = a.mat.Inverse(fes.FreeDofs(), inverse="sparsecholesky")
    gfu.vec.data = inv * f.vec

# 7. 可视化
# 在 Jupyter/VS Code 中直接运行会显示一个交互式 3D 窗口
# Draw(gfu, mesh, "ElectricField")
# 也可以查看模长
Draw(Norm(gfu), mesh, "Abs_E")
# 定义切面参数：x, y, z 是切面的法线方向，dist 是切平面的偏移距离


WebGuiWidget(layout=Layout(height='500px', width='100%'), value={'gui_settings': {}, 'ngsolve_version': '6.2.2…

BaseWebGuiScene

In [3]:
# 修改后的 VTK 导出部分

vtk = VTKOutput(mesh, 
                coefs=[Norm(gfu)], 
                names=["E_norm"], 
                filename="maxwell_result", 
                subdivision=2)
vtk.Do()

print("导出成功！请在 ParaView 中查看 E_real (实部) 或 E_norm (模长)。")


导出成功！请在 ParaView 中查看 E_real (实部) 或 E_norm (模长)。
